# N-ATLaS endpoint for natlas-kit (free Colab GPU)

This downloads N-ATLaS and serves it with an OpenAI-compatible API, then opens a public URL.
Give that URL to natlas-kit (`NATLAS_BACKEND=openai`, `NATLAS_ENDPOINT=<url>`, `NATLAS_MODEL=natlas`).

**One-time setup (do this once, before the first run):**
1. Get a free ngrok token: https://dashboard.ngrok.com/get-started/your-authtoken
2. In this notebook, click the **key icon (🔑)** in the left sidebar → **Add new secret**.
   Name it `NGROK_TOKEN`, paste your token as the value, and toggle **Notebook access ON**.

**Then every time:**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. **Runtime → Run all** (or run the 3 cells in order).
3. Cell 3 prints an `ENDPOINT URL` — copy it and send it over. Keep this tab open while testing.


In [ ]:
# Cell 1 - install ollama + GPU detection tools, and download the N-ATLaS GGUF (ungated, no token needed)
!apt-get -qq install -y zstd pciutils lshw   # zstd for ollama's installer; pciutils/lshw so it detects the T4 GPU
!curl -fsSL https://ollama.com/install.sh | sh
from huggingface_hub import hf_hub_download
MODEL = hf_hub_download('QuantFactory/N-ATLaS-GGUF', 'N-ATLaS.Q4_K_M.gguf')
with open('Modelfile','w') as f:
    f.write(f'FROM {MODEL}\nPARAMETER temperature 0.1\nPARAMETER repeat_penalty 1.12\n')
print('model ready at', MODEL)

In [ ]:
# Cell 2 - serve N-ATLaS on the GPU (pciutils/lshw from Cell 1 means it's detected from the start)
import subprocess, time, os
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
print('creating model (loads the GGUF)...')
subprocess.run(['ollama', 'create', 'natlas', '-f', 'Modelfile'])
print(subprocess.run(['ollama', 'run', 'natlas', 'Reply with: OK'], capture_output=True, text=True).stdout)
print('GPU memory in use (should be non-zero if the GPU engaged):')
os.system("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv")

In [ ]:
# Cell 3 - expose N-ATLaS via ngrok (Cloudflare quick-tunnels block API traffic, so we don't use them)
# Uses the NGROK_TOKEN secret from the one-time setup above - no token pasted in this file.
!pip -q install pyngrok
from pyngrok import ngrok, conf

token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_TOKEN')
except Exception:
    pass

if not token:
    raise RuntimeError(
        "No NGROK_TOKEN secret found. Click the key icon in the left sidebar, add a secret "
        "named NGROK_TOKEN with your token from https://dashboard.ngrok.com/get-started/your-authtoken, "
        "turn on Notebook access, then re-run this cell."
    )

conf.get_default().auth_token = token
ngrok.kill()  # clear any old tunnel
public = ngrok.connect(11434, "http")
print("\n==================== ENDPOINT URL ====================")
print(public.public_url)
print("Use: NATLAS_BACKEND=openai  NATLAS_ENDPOINT=" + public.public_url + "  NATLAS_MODEL=natlas")
print("=====================================================\n")
print("Keep this tab open while testing.")

In [ ]:
# Cell 4 - run natlas-kit's real tests against N-ATLaS, from inside Colab (localhost,
# no tunnel needed - ngrok's free tier blocks external API traffic from Colab IPs).
!git clone -q https://github.com/nevodesigns/natlas-kit.git
%cd natlas-kit
!pip install -q -e .
import os
os.environ["NATLAS_BACKEND"] = "openai"
os.environ["NATLAS_ENDPOINT"] = "http://localhost:11434"
os.environ["NATLAS_MODEL"] = "natlas"
!natlas ping
!natlas ask "Translate 'good morning' into Yoruba. Reply with only the translation."
!natlas eval evals/suite_core.json | tee /content/natlas_eval_report.md
!python examples/egotrack_voice_to_ledger.py